In [8]:
import cv2
import os
import numpy as np
from PIL import Image
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import joblib

# --- CONFIGURATION ---
RADIUS = 1
NEIGHBORS = 8
GRID_X = 8
GRID_Y = 8

# Path to your dataset
DATA_PATH = r'C:\Users\hp\Desktop\Attendance-System-Using-Face-Recognition\Dataset\training\Cleaned_Training'

In [9]:
def rotate_image(image, angle):
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(image, M, (w, h))
    return rotated

def extract_hog_features(image):
    """
    Extracts Histogram of Oriented Gradients (HOG) features.
    """
    features = hog(image, 
                   orientations=9, 
                   pixels_per_cell=(8, 8), 
                   cells_per_block=(2, 2), 
                   block_norm='L2-Hys', 
                   visualize=False)
    return features

In [10]:
def load_and_augment_data(path):
    image_paths = [os.path.join(path, f) for f in os.listdir(path)]
    
    # Lists for LBPH (Raw Images)
    lbph_faces = []
    lbph_ids = []
    
    # Lists for SVM & KNN (HOG Feature Vectors)
    hog_features = []
    hog_labels = []
    
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

    print(f"Processing {len(image_paths)} source images...")
    
    for img_path in image_paths:
        try:
            img = Image.open(img_path).convert('L')
            img_np = np.array(img, 'uint8')
            
            # CAUTION: Ensure filenames are "User.ID.jpg"
            user_id = int(os.path.split(img_path)[-1].split(".")[1])
            
            # Base Preprocessing (200x200 standard)
            face_resized = cv2.resize(img_np, (200, 200), interpolation=cv2.INTER_CUBIC)
            face_smooth = cv2.bilateralFilter(face_resized, 5, 75, 75)
            enhanced = clahe.apply(face_smooth)
            
            # --- AUGMENTATION STACK ---
            aug_imgs = [enhanced]
            aug_imgs.append(cv2.flip(enhanced, 1))                        # Flip
            aug_imgs.append(rotate_image(enhanced, -10))                  # Rotate Left
            aug_imgs.append(rotate_image(enhanced, 10))                   # Rotate Right
            aug_imgs.append(cv2.convertScaleAbs(enhanced, alpha=1, beta=-40)) # Darker
            aug_imgs.append(cv2.convertScaleAbs(enhanced, alpha=1, beta=40))  # Brighter
            
            # Add to datasets
            for face in aug_imgs:
                # 1. For LBPH: Add the raw image
                lbph_faces.append(face)
                lbph_ids.append(user_id)
                
                # 2. For SVM & KNN: Extract HOG features
                feat = extract_hog_features(face)
                hog_features.append(feat)
                hog_labels.append(user_id)
                
        except Exception as e:
            print(f"Skipping {img_path}: {e}")
            
    return lbph_faces, lbph_ids, hog_features, hog_labels

# --- EXECUTE LOAD ---
faces, ids, hog_feats, hog_lbls = load_and_augment_data(DATA_PATH)

if len(faces) > 0:
    print(f"✅ Data Loaded successfully.")
    print(f"Total Augmented Samples: {len(faces)}")
else:
    print("⚠️ No data found. Check your path.")

Processing 139 source images...
✅ Data Loaded successfully.
Total Augmented Samples: 834


In [11]:
if len(faces) > 0:
    print("Training LBPH Model (OpenCV)...")
    lbph = cv2.face.LBPHFaceRecognizer_create(radius=RADIUS, neighbors=NEIGHBORS, grid_x=GRID_X, grid_y=GRID_Y)
    lbph.train(faces, np.array(ids))
    lbph.save('trainer.yml')
    print("✅ Saved 'trainer.yml'")
else:
    print("Skipping LBPH: No data.")

Training LBPH Model (OpenCV)...
✅ Saved 'trainer.yml'


In [ ]:
if len(hog_feats) > 0:
    print("Training SVM Model (HOG)...")
    # SVM Training
    svm = SVC(kernel='linear', C=10.0, gamma='scale', probability=True, random_state=42)
    svm.fit(hog_feats, hog_lbls)
    joblib.dump(svm, 'svm_face_model.pkl')
    print("✅ Saved 'svm_face_model.pkl'")
else:
    print("Skipping SVM: No data.")

Training SVM Model (HOG)...


In [ ]:
if len(hog_feats) > 0:
    print(f"Starting KNN Training on {len(hog_feats)} HOG samples...")

    # Metric: Euclidean works well for HOG
    # Weights: 'distance' ensures closer neighbors count more
    knn_hog = KNeighborsClassifier(n_neighbors=5, metric='euclidean', weights='distance')

    print("Fitting KNN on HOG features...")
    knn_hog.fit(hog_feats, hog_lbls)
    
    joblib.dump(knn_hog, 'knn_hog_model.pkl')
    print("✅ Saved 'knn_hog_model.pkl'")
else:
    print("Skipping KNN: No data.")

Starting KNN Training on 834 HOG samples...
Fitting KNN on HOG features...
✅ Saved 'knn_hog_model.pkl'


In [ ]:
import cv2
import numpy as np
import os
import joblib
from PIL import Image as PILImage
from skimage.feature import hog
from sklearn.neighbors import KNeighborsClassifier

# --- CONFIGURATION ---
DATA_PATH = r'C:\Users\hp\Desktop\Attendance-System-Using-Face-Recognition\Dataset\training\Cleaned_Training'

# --- 1. LOAD PRE-TRAINED MODELS ---
models = {}
print("Loading system...")

try:
    lbph = cv2.face.LBPHFaceRecognizer_create(radius=1, neighbors=8, grid_x=8, grid_y=8)
    lbph.read('trainer.yml')
    models['LBPH'] = lbph
    print("✅ LBPH Model Loaded.")
except: print("❌ LBPH Model missing (trainer.yml)")

try:
    models['SVM'] = joblib.load('svm_face_model.pkl')
    print("✅ SVM Model Loaded.")
except: print("❌ SVM Model missing (svm_face_model.pkl)")

try:
    models['KNN'] = joblib.load('knn_hog_model.pkl')
    print("✅ KNN Model Loaded.")
except: print("❌ KNN Model missing (knn_hog_model.pkl)")

# --- 2. LOAD TRAINING DATA (Required for Restricted KNN) ---
hog_feats, hog_lbls = [], []

try:
    # Try loading fast from file
    hog_feats, hog_lbls = joblib.load('hog_features.pkl')
    print(f"✅ Training Features Loaded ({len(hog_feats)} samples).")
except:
    print("⚠️ Features file not found. Generating from images (One-time setup)...")
    if os.path.exists(DATA_PATH):
        image_paths = [os.path.join(DATA_PATH, f) for f in os.listdir(DATA_PATH)]
        for path in image_paths:
            try:
                img = PILImage.open(path).convert('L')
                uid = int(os.path.split(path)[-1].split(".")[1])
                img = img.resize((200, 200), PILImage.LANCZOS)
                # Extract HOG
                feat = hog(np.array(img), orientations=9, pixels_per_cell=(8, 8), 
                           cells_per_block=(2, 2), block_norm='L2-Hys', visualize=False)
                hog_feats.append(feat)
                hog_lbls.append(uid)
            except: pass
        
        # Save so we don't do this again
        joblib.dump((hog_feats, hog_lbls), 'hog_features.pkl')
        print("✅ Features generated and saved to 'hog_features.pkl'.")
    else:
        print(f"❌ Error: Dataset path not found: {DATA_PATH}")

print("------------------------------------------------")
if len(models) == 3 and len(hog_feats) > 0:
    print("🚀 SYSTEM READY FOR STACKING.")
else:
    print("⚠️ SYSTEM INCOMPLETE. Check errors above.")

Loading system...
✅ LBPH Model Loaded.
✅ SVM Model Loaded.
✅ KNN Model Loaded.
✅ Training Features Loaded (139 samples).
------------------------------------------------
🚀 SYSTEM READY FOR STACKING.


In [ ]:
import numpy as np
from collections import Counter
from sklearn.neighbors import KNeighborsClassifier
from skimage.feature import hog

class StackedFaceRecognizer:
    def __init__(self, models_dict, train_feats, train_lbls):
        self.lbph = models_dict['LBPH']
        self.svm = models_dict['SVM']
        self.knn = models_dict['KNN']
        self.X_train = np.array(train_feats)
        self.y_train = np.array(train_lbls)
        
        # Track IDs already predicted by each model in the current frame
        self.session_history = {
            'SVM': set(),
            'KNN': set(),
            'LBPH': set()
        }
        
    def reset_session(self):
        """Call this at the start of every new image frame."""
        self.session_history['SVM'].clear()
        self.session_history['KNN'].clear()
        self.session_history['LBPH'].clear()

    def extract_hog(self, image):
        return hog(image, orientations=9, pixels_per_cell=(8, 8), 
                   cells_per_block=(2, 2), block_norm='L2-Hys', visualize=False)

    def predict_cleaned(self, face_img, svm_pred, knn_pred, lbph_pred):
        valid_votes = []
        
        # 1. Process SVM Vote
        if svm_pred:
            s_id = svm_pred['id']
            if s_id not in self.session_history['SVM']:
                valid_votes.append(s_id)
                self.session_history['SVM'].add(s_id)
            # If s_id is already in history, it is ignored (discarded)

        # 2. Process KNN Vote
        if knn_pred:
            k_id = knn_pred['id']
            if k_id not in self.session_history['KNN']:
                valid_votes.append(k_id)
                self.session_history['KNN'].add(k_id)

        # 3. Process LBPH Vote
        if lbph_pred:
            l_id = lbph_pred['id']
            if l_id not in self.session_history['LBPH']:
                valid_votes.append(l_id)
                self.session_history['LBPH'].add(l_id)
        
        # Check if we have any valid votes left after filtering duplicates
        if not valid_votes: 
            return None, 0.0, "No Unique Votes"
        
        # Consensus logic
        top_vote, count = Counter(valid_votes).most_common(1)[0]
        
        if count >= 2:
            return top_vote, 1.0, f"Voting ({count}/3)"

        # Fallback: Restricted KNN using only candidates proposed by models
        valid_candidates = list(set(valid_votes))
        hog_vec = self.extract_hog(face_img)
        mask = np.isin(self.y_train, valid_candidates)
        
        if np.sum(mask) == 0: 
            return None, 0.0, "Err"
        
        n_neighbors = min(5, len(self.X_train[mask]))
        mini_knn = KNeighborsClassifier(n_neighbors=n_neighbors, metric='euclidean', algorithm='brute')
        mini_knn.fit(self.X_train[mask], self.y_train[mask])
        
        final_id = mini_knn.predict([hog_vec])[0]
        final_conf = np.max(mini_knn.predict_proba([hog_vec])[0])
        
        return final_id, final_conf, "Restricted KNN"

if 'models' in locals() and 'hog_feats' in locals() and len(models) == 3:
    stacker = StackedFaceRecognizer(models, hog_feats, hog_lbls)
    print("✅ System Ready with Duplicate Model Filtering.")
else:
    print("⚠️ Models not loaded.")

⚠️ Models not loaded.
